<a href="https://colab.research.google.com/github/jeyajeevaj17/automatic_message_cipher/blob/main/AutomaticMessageCipher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import gradio as gr
import random
import hashlib
import os

# --- Emoji Base Map ---
char_to_emoji_bases = {
    "Standard": {
        'a': '😀', 'b': '😂', 'c': '😎', 'd': '😍', 'e': '🤔',
        'f': '🙄', 'g': '😴', 'h': '😡', 'i': '🤩', 'j': '🥳',
        'k': '😇', 'l': '😜', 'm': '🤖', 'n': '👻', 'o': '💩',
        'p': '👽', 'q': '👾', 'r': '🎃', 's': '🐵', 't': '🦄',
        'u': '🐸', 'v': '🦊', 'w': '🐶', 'x': '🐱', 'y': '🦁', 'z': '🐯',
        '0': '🍎', '1': '🚀', '2': '🎈', '3': '🌟', '4': '🍕',
        '5': '🐢', '6': '🎩', '7': '⚡', '8': '🍄', '9': '🎲',
        ' ': '⬜', '.': '⚫', ',': '⚪', '!': '❗', '?': '❓', '-': '➖',
        '_': '⬛', '\'': '🟤', '"': '🟠', '@': '📧', '#': '🔢',
        '%': '📊', '&': '➕', '(': '⭕', ')': '⭕', '[': '〚', ']': '〛',
        '{': '❴', '}': '❵', ':': '⏰', ';': '💉', '+': '➕', '/': '➗',
        '\\': '🛡️', '=': '🟰', '*': '✳️', '<': '◀️', '>': '▶️',
        '~': '〰️', '^': '⬆️', '$': '💲',
    }
}

UPPERCASE_MARKER = '🔺'
ALL_CHARS = list(char_to_emoji_bases["Standard"].keys())

# --- Cipher Generation ---
def generate_cipher(theme, password=None):
    base_map = char_to_emoji_bases.get(theme, char_to_emoji_bases["Standard"])
    emojis = list(base_map.values())
    chars = ALL_CHARS[:]

    # Randomize or deterministic
    if password:
        seed = int(hashlib.sha256((password + theme).encode()).hexdigest(), 16)
    else:
        seed = random.randint(0, 1_000_000_000)

    rnd = random.Random(seed)
    shuffled_emojis = emojis[:]
    rnd.shuffle(shuffled_emojis)

    cipher_map = {c: e for c, e in zip(chars, shuffled_emojis)}
    inverse_map = {v: k for k, v in cipher_map.items()}
    return cipher_map, inverse_map

# --- Encryption / Decryption ---
def encrypt_text(text, cipher_map):
    result = []
    for ch in text:
        if ch.isupper() and ch.lower() in cipher_map:
            result.append(UPPERCASE_MARKER + cipher_map[ch.lower()])
        elif ch in cipher_map:
            result.append(cipher_map[ch])
        else:
            result.append(ch)
    return "".join(result)

def decrypt_text(text, inverse_map):
    i = 0
    result = []
    while i < len(text):
        if text[i:i+len(UPPERCASE_MARKER)] == UPPERCASE_MARKER:
            emoji = text[i+len(UPPERCASE_MARKER)]
            ch = inverse_map.get(emoji, '')
            result.append(ch.upper())
            i += len(UPPERCASE_MARKER) + 1
        else:
            ch = inverse_map.get(text[i], '')
            if ch:
                result.append(ch)
            else:
                result.append(text[i])
            i += 1
    return "".join(result)

def is_emoji_encrypted(text, inverse_map):
    for emoji in inverse_map.keys():
        if emoji in text:
            return True
    return UPPERCASE_MARKER in text

def process_text(input_text, mode, theme, password):
    if not input_text:
        return "", "No input provided."
    cipher_map, inverse_map = generate_cipher(theme, password)
    if mode == "Auto":
        mode_use = "Decrypt" if is_emoji_encrypted(input_text, inverse_map) else "Encrypt"
    else:
        mode_use = mode.capitalize()
    if mode_use == "Encrypt":
        output = encrypt_text(input_text, cipher_map)
    else:
        output = decrypt_text(input_text, inverse_map)
    return output, f"{mode_use}ion done."

# --- File Processing ---
def process_file(file, mode, theme, password):
    if file is None:
        return "", "No file uploaded.", None
    try:
        with open(file.name, "r", encoding="utf-8") as f:
            content = f.read()

        output, status = process_text(content, mode, theme, password)
        new_filename = f"processed_{os.path.basename(file.name)}"
        with open(new_filename, "w", encoding="utf-8") as f:
            f.write(output)
        return output, status, new_filename
    except Exception as e:
        return "", f"File processing error: {e}", None

# --- Cipher Display ---
def generate_cipher_text(theme, password):
    cipher_map, _ = generate_cipher(theme, password)
    lines = [f"{ch if ch != ' ' else 'SPACE'} → {cipher_map[ch]}" for ch in sorted(cipher_map.keys())]
    return "\n".join(lines)

# --- UI Construction ---
with gr.Blocks() as demo:
    gr.Markdown("# 🔐 **Emoji Cipher Encoder/Decoder (Full Version)**")

    # --- TEXT MODE TAB ---
    with gr.Tab("Text Mode"):
        with gr.Row():
            theme = gr.Dropdown(list(char_to_emoji_bases.keys()), label="Emoji Theme", value="Standard")
            password = gr.Textbox(label="Password (optional)", placeholder="Enter password for consistent cipher")
            mode = gr.Radio(["Auto", "Encrypt", "Decrypt"], label="Mode", value="Auto")

        input_text = gr.Textbox(label="Input Text", lines=6, placeholder="Enter text or emoji cipher...")
        output_text = gr.Textbox(label="Output Text", lines=6, interactive=False)
        status = gr.Textbox(label="Status", interactive=False)

        with gr.Row():
            run_btn = gr.Button("▶️ Run")
            clear_btn = gr.Button("🧹 Clear")

        run_btn.click(process_text, [input_text, mode, theme, password], [output_text, status])
        clear_btn.click(lambda: ("", "", ""), None, [input_text, output_text, status])

    # --- FILE MODE TAB ---
    with gr.Tab("File Mode"):
        gr.Markdown("### 📂 Encrypt or Decrypt Entire Files")

        with gr.Row():
            theme_file = gr.Dropdown(list(char_to_emoji_bases.keys()), label="Emoji Theme", value="Standard")
            password_file = gr.Textbox(label="Password (optional)")
            mode_file = gr.Radio(["Auto", "Encrypt", "Decrypt"], label="Mode", value="Auto")

        file_input = gr.File(label="Upload UTF-8 Text File")
        file_output = gr.Textbox(label="Processed File Preview", lines=6, interactive=False)
        file_status = gr.Textbox(label="Status", interactive=False)
        output_file = gr.File(label="Download Processed File")

        with gr.Row():
            run_file_btn = gr.Button("▶️ Run File")
            clear_file_btn = gr.Button("🧹 Clear File")

        # Run manually after upload + mode select
        run_file_btn.click(process_file, [file_input, mode_file, theme_file, password_file],
                           [file_output, file_status, output_file])
        clear_file_btn.click(lambda: ("", "", None, ""), None,
                             [file_output, file_status, output_file, file_input])

    # --- CIPHER VIEWER TAB ---
    with gr.Tab("Cipher Viewer"):
        gr.Markdown("### 🔍 View Generated Cipher Mapping")
        cipher_display = gr.Textbox(label="Generated Cipher (Char → Emoji)", lines=10, interactive=False)
        generate_btn = gr.Button("🔄 Generate Cipher")
        clear_cipher_btn = gr.Button("🧹 Clear Cipher")

        generate_btn.click(generate_cipher_text, [theme, password], [cipher_display])
        clear_cipher_btn.click(lambda: "", None, [cipher_display])

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9730ffbd6d1e1fbd19.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
